# Linear Regression


# Introduction

## Introduction

[![](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tools4ds/DS701-Materials-FA26/blob/main/jupyter_notebooks/10-Regression-I-Linear.ipynb)

In [ ]:
#| echo: false
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import laUtilities as ut

%matplotlib inline

import statsmodels.api as sm

def centerAxes(ax):
    ax.spines['left'].set_position('zero')
    ax.spines['right'].set_color('none')
    ax.spines['bottom'].set_position('zero')
    ax.spines['top'].set_color('none')
    ax.xaxis.set_ticks_position('bottom')
    ax.yaxis.set_ticks_position('left')
    bounds = np.array([ax.axes.get_xlim(), ax.axes.get_ylim()])
    ax.plot(bounds[0][0],bounds[1][0],'')
    ax.plot(bounds[0][1],bounds[1][1],'')

![](figs/Sir_Francis_Galton_by_Charles_Wellington_Furse.jpg)

[Sir Francis Galton by Charles Wellington Furse](https://commons.wikimedia.org/wiki/File:Sir_Francis_Galton_by_Charles_Wellington_Furse.jpg#/media/File:Sir_Francis_Galton_by_Charles_Wellington_Furse.jpg)

<!--
[National Portrait Gallery](https://en.wikipedia.org/wiki/National_Portrait_Gallery,_London)
[NPG 3916](http://www.npg.org.uk/collections/search/portrait.php?search=ap&amp;npgno=3916&amp;eDate=&amp;lDate=)
-->


![](figs/galton-title.png)

In 1886 Francis Galton published his observations about the height of children
compared to their parents.


## Regression to the Mean



![](figs/galton-regression.png)




Defined mid-parent as the average of the heights of the parents.

When parents are taller or shorter than average, their children are more likely
to be closer to the average height than their parents. 


So called **"regression to the mean."**

Galton fit a straight line to this effect, and the fitting of lines or curves to
data has come to be called regression as well.




## Praise versus Punishment




![](figs/L17-Israeli-Airforce-Pirate.png)

<!-- From https://airpowerasia.com/2020/08/30/israels-giora-hawkeye-epstein-ace-of-aces-of-supersonic-fighter-jets-17-aerial-victories/ -->

![Daniel Kahneman](figs/L17-Daniel-Kahneman--NYT.png)




In the 60s, psychologist Daniel Kahneman studied the effect of praise versus
punishment on performance of Israeli Air Force trainees. He was led to believe
that praise was more effective than punishment.

Air force trainers disagreed and said that when they praised pilots that did well,
they usually did worse the next time, and when they punished pilots that did poorly,
they usually did better the next time.

What Kahneman later realised was that this phenomenon could be explained by
the idea of regression to the mean.

If a pilot does worse than average on a task, they are more likely to do better
the next time, and if they do better than average, they are more likely to do
worse the next time.



## Firing Coaches

![NFL Coaches](figs/L17-nfl-coaching-candidates_0jpg.jpg)

[Reference](https://faculty.mccombs.utexas.edu/carlos.carvalho/teaching/regression_to_the_mean.pdf)

Is it a good strategy to fire a coach after a bad season?

> bad season -> fire and replace coach -> better season next year

When this could be interpreted as:

> bad season (outlier) -> regression to the mean -> better season next year


## This applies to many situations

> "I had so many colds last year, and then I started herbal remedies, and I have much
> fewer colds this year."

> "My investments were doing quite bad last year, so switched investment managers, and
> now it is doing much better."

> "The team had a terrible season, so we fired the coach -- and this year they're
> winning again."

<br>
When in fact, they could be (at least partially) explained by:

**Regression to the mean**



The praise-versus-punishment and coaching stories above are two famous
instances; Kahneman's version is in *Thinking, Fast and Slow*. We will meet the
effect once more at the end of this lecture, where it falls out of the algebra
of least squares.


## Model Types

The most common form of machine learning is __regression__, which means
constructing an equation that describes the relationships among variables.

It is a form of supervised learning: 

* whereas __classification__ deals with predicting categorical features (labels or classes), 
* __regression__ deals with predicting continuous features (real values).

## Fit to Linear Models

For example, we may look at these points and decide to model them using a line.

In [ ]:
#| fig-align: center
ax = plt.figure(figsize = (5, 5)).add_subplot()
centerAxes(ax)
line = np.array([1, 0.5])
np.random.seed(1)
xlin = -10.0 + 20.0 * np.random.random(100)
ylin = line[0] + (line[1] * xlin) + np.random.randn(100)
ax.plot(xlin, ylin, 'ro', markersize = 4)
plt.show()

## Fit to Quadratic Models

We may look at these points and decide to model them using a quadratic function.

In [ ]:
#| fig-align: center
ax = plt.figure(figsize = (5, 5)).add_subplot()
centerAxes(ax)
quad = np.array([1, 3, 0.5])
std = 8.0
np.random.seed(1)
xquad = -10.0 + 20.0 * np.random.random(100)
yquad = quad[0] + (quad[1] * xquad) + (quad[2] * xquad * xquad) + std * np.random.randn(100)
ax.plot(xquad, yquad, 'ro', markersize = 4)
plt.show()

## Fit to Logarithmic Models

And we may look at these points and decide to model them using a logarithmic function.

In [ ]:
#| fig-align: center
ax = plt.figure(figsize = (5, 5)).add_subplot()
centerAxes(ax)
log = np.array([1, 4])  
std = 1.5
np.random.seed(1)
xlog = 10.0 * np.random.random(100)
ylog = log[0] + log[1] * np.log(xlog) + std * np.random.randn(100)
ax.plot(xlog, ylog, 'ro', markersize=4)
plt.show()

## Approach

Clearly, none of these datasets agrees perfectly with the proposed model.   

So the question arises:

How do we find the __best__ linear function (or quadratic function, or logarithmic function) given the data?

# Framework

## Framework

This problem has been studied extensively in the field of statistics.   

Certain terminology is used:

* Some values are referred to as "independent," and
* Some values are referred to as "dependent."


The basic regression task is:

* given a set of independent variables 
* and the associated dependent variables, 
* estimate the parameters of a model (such as a line, parabola, etc) that describes
  how the dependent variables are related to the independent variables.



<br>

The **independent variables** are collected into a matrix $X,$ sometimes called the
__design matrix.__

The dependent variables are collected into an __observation__ vector $\mathbf{y}.$

The parameters of the model (for any kind of model) are collected into a 
__parameter__ vector $\boldsymbol\beta.$


In [ ]:
#| fig-align: center
ax = plt.figure(figsize = (5, 5)).add_subplot()
centerAxes(ax)
line = np.array([1, 0.5])
np.random.seed(1)
xlin = -10.0 + 20.0 * np.random.random(100)
ylin = line[0] + (line[1] * xlin) + np.random.randn(100)
ax.plot(xlin, ylin, 'ro', markersize = 4)
ax.plot(xlin, line[0] + line[1] * xlin, 'b-')
plt.text(-9, 3, r'$y = \beta_0 + \beta_1x$', size=20)
plt.show()

## Least-Squares Lines

* The first kind of model we'll study is a linear equation, $y = \beta_0 + \beta_1 x.$

* Experimental data often produce points $(x_1, y_1), \dots, (x_n, y_n)$ that seem
to lie close to a line.   

* We want to determine the parameters $\beta_0, \beta_1$ that define a line that is
as "close" to the points as possible.



Suppose we have a line $y = \beta_0 + \beta_1 x$.   

For each data point $(x_j, y_j),$ there is a point $(x_j, \beta_0 + \beta_1 x_j)$
that is the point on the line with the same $x$-coordinate.

In [ ]:
#| fig-align: center
def plot_residuals(x, y, beta_0, beta_1, figsize=(7, 4), title=None, legend=True):
    # scatter the data, draw the line y = beta_0 + beta_1 x, and mark each residual
    y_line = beta_0 + beta_1 * x

    fig, ax = plt.subplots(figsize=figsize)
    ax.scatter(x, y, color='blue', label='Data points', zorder=5)
    ax.scatter(x, y_line, color='red', label='Predicted values', zorder=5)
    ax.plot(x, y_line, color='cyan', label='Regression line (y = β0 + β1x)', zorder=4)

    # residuals: vertical segments from each point to the line
    for i in range(len(x)):
        ax.vlines(x[i], y_line[i], y[i], color='blue', linestyles='dashed',
                  label='Residual' if i == 0 else "", zorder=2)

    # label observed and predicted values, alternating above/below
    y_offset = [0.2, -0.2, 0.2, -0.2, 0.2]
    y_line_offset = [-0.3, 0.3, -0.3, 0.3, -0.2]
    for i in range(len(x)):
        ax.text(x[i], y[i] + y_offset[i], f'({x[i]}, {y[i]})', fontsize=9, ha='center')
        ax.text(x[i], y_line[i] + y_line_offset[i], f'({x[i]}, {y_line[i]:.1f})', fontsize=9, ha='center')

    ax.spines['top'].set_color('none')
    ax.spines['right'].set_color('none')
    ax.xaxis.set_ticks([])
    ax.yaxis.set_ticks([])
    if title:
        ax.set_title(title)
    if legend:
        ax.legend()
    return ax

x = np.array([1, 2, 3, 4, 5])
y = np.array([3.7, 2.0, 2.1, 0.1, 1.5])
plot_residuals(x, y, beta_0=4, beta_1=-0.8, title='Linear Regression with Residuals')
plt.show()

In [ ]:
#| echo: false
plot_residuals(x, y, beta_0=4, beta_1=-0.8, figsize=(4, 3), legend=False)
plt.show()

We call 

* $y_j$ the __observed__ value of $y$ and 
* $\beta_0 + \beta_1 x_j$ the __predicted__ $y$-value.   


The difference between an observed $y$-value and a predicted $y$-value is called
a __residual__.


There are several ways to measure how "close" the line is to the data. 


The usual choice is to sum the squares of the residuals.  


The __least-squares line__ is the line $y = \beta_0 + \beta_1x$ that minimizes
the sum of squares of the residuals. 

The coefficients $\beta_0, \beta_1$ of the line are called
__regression coefficients.__


## A least-squares problem

If the data points were on the line, the parameters $\beta_0$ and $\beta_1$ would satisfy the equations

$$
\begin{align*}
\beta_0 + \beta_1 x_1 &= y_1, \\
\beta_0 + \beta_1 x_2 &= y_2, \\
\beta_0 + \beta_1 x_3 &= y_3, \\
&\vdots \\
\beta_0 + \beta_1 x_n &= y_n.
\end{align*}
$$

--- 

We can write this system as 

$$
X\boldsymbol\beta = \mathbf{y},
$$

where 

$$
X=
\begin{bmatrix}
1 & x_1 \\
1 & x_2 \\
\vdots & \vdots \\
1 & x_n
\end{bmatrix},
\;\;\boldsymbol\beta = \begin{bmatrix}\beta_0\\\beta_1\end{bmatrix},
\;\;\mathbf{y}=\begin{bmatrix}y_1\\y_2\\\vdots\\y_n\end{bmatrix}.
$$

---

Of course, if the data points don't actually lie exactly on a line, 

... then there are no parameters $\beta_0, \beta_1$ for which the predicted
$y$-values in $X\boldsymbol\beta$ equal the observed $y$-values in $\mathbf{y}$, 

... and $X\boldsymbol\beta=\mathbf{y}$ has no solution.

---

Now, since the data doesn't fall exactly on a line, we have decided to seek the
$\boldsymbol\beta$ that minimizes the sum of squared residuals, i.e.,

$$
\sum_i (\beta_0 + \beta_1 x_i - y_i)^2
=\Vert X\boldsymbol\beta -\mathbf{y}\Vert^2.
$$

This is key: 

**Important**

__The sum of squares of the residuals__ is __exactly__ the 
__square of the distance between the vectors $X\boldsymbol\beta$ and $\mathbf{y}.$__


---

Computing the least-squares solution of $X\boldsymbol\beta = \mathbf{y}$ is equivalent to
finding the $\boldsymbol\beta$ that determines the least-squares line.

In [ ]:
#| fig-align: center
ax = plt.figure(figsize = (5, 5)).add_subplot()
centerAxes(ax)
line = np.array([1, 0.5])
np.random.seed(1)
xlin = -10.0 + 20.0 * np.random.random(100)
ylin = line[0] + (line[1] * xlin) + np.random.randn(100)
ax.plot(xlin, ylin, 'ro', markersize = 4)
ax.plot(xlin, line[0] + line[1] * xlin, 'b-')
plt.text(-9, 3, r'$y = \beta_0 + \beta_1x$', size=20)
plt.show()

---

Now, to obtain the least-squares line, find the least-squares solution to
$X\boldsymbol\beta = \mathbf{y}.$

Where does that solution come from? Not from a formula to memorize -- from a
picture.

**Prerequisites**

The next two slides use the column space of a matrix, orthogonal projection,
and the normal equations. If these are rusty, read
[Least squares](A7-Linear-Algebra-Refresher.qmd#least-squares) in the
linear-algebra refresher first; its
[Least squares beware](A7-Linear-Algebra-Refresher.qmd#least-squares-beware)
page is also worth a look before you write `np.linalg.inv(X.T @ X)` yourself.


## Least squares is a projection


![](figs/L10-recap-projection.png)


* As $\boldsymbol\beta$ varies, $X\boldsymbol\beta$ ranges over the
  **column space** $\text{Col}(X)$: all linear combinations of the columns of $X$.
* We want the point $\hat{\mathbf{y}} = X\hat{\boldsymbol\beta}$ of $\text{Col}(X)$
  that is closest to $\mathbf{y}$.
* The closest point of a subspace is the **orthogonal projection** of $\mathbf{y}$
  onto it. So the residual $\mathbf{e} = \mathbf{y} - \hat{\mathbf{y}}$ is
  perpendicular to the subspace, hence to **every column** of $X$:
  $$X^T(\mathbf{y} - X\hat{\boldsymbol\beta}) = \mathbf{0}.$$




Rearranging $X^T(\mathbf{y} - X\hat{\boldsymbol\beta}) = \mathbf{0}$ gives the
__normal equations__:

$$
X^TX\hat{\boldsymbol\beta} = X^T\mathbf{y}.
$$

They say exactly this: the residual $X\hat{\boldsymbol\beta} - \mathbf{y}$ is
normal (i.e. orthogonal) to the column space of $X$.

The normal equations __always__ have at least one solution ($X^T\mathbf{y}$
lies in the column space of $X^T$, which equals the column space of $X^TX$).

If $X^TX$ is invertible, the solution is unique:

$$
\hat{\boldsymbol\beta} = (X^TX)^{-1} X^T\mathbf{y}.
$$


## The hat matrix

Substitute $\hat{\boldsymbol\beta}$ back into $\hat{\mathbf{y}} = X\hat{\boldsymbol\beta}$:

$$
\hat{\mathbf{y}} = X(X^TX)^{-1}X^T\mathbf{y} = P\mathbf{y}.
$$

$P = X(X^TX)^{-1}X^T$ is the __projection matrix__, or __hat matrix__ (it puts
the hat on $\mathbf{y}$). It depends only on $X$, not on $\mathbf{y}$.

* $P$ is **symmetric** ($P^T = P$) and **idempotent** ($P^2 = P$): projecting
  twice changes nothing.
* $\operatorname{tr} P = p$, the number of columns of $X$: the fit "uses up" $p$
  degrees of freedom.
* The residual is $\mathbf{e} = (I - P)\mathbf{y}$, and
  $\hat{\mathbf{y}}^T\mathbf{e} = \mathbf{y}^TP(I-P)\mathbf{y} = 0$: fitted
  values and residuals are orthogonal. That is Pythagoras,
  $\|\mathbf{y}\|^2 = \|\hat{\mathbf{y}}\|^2 + \|\mathbf{e}\|^2$, and it becomes
  $\text{TSS} = \text{RSS} + \text{ESS}$ below.
* The diagonal entry $P_{ii}$ is the **leverage** of observation $i$: how hard
  $y_i$ pulls on its own fitted value. It returns when we look at residuals.


## When is $X^TX$ invertible?

For $X$ an $n \times p$ matrix ($n$ observations, $p$ columns):

$$
X^TX \text{ is invertible} \iff X \text{ has full column rank, i.e. } \operatorname{rank}(X) = p,
$$

meaning no column of $X$ is a linear combination of the others.

This requires $n \geq p$ (at least as many observations as columns), but
$n \ge p$ alone is not enough: two identical columns make $X^TX$ singular at
any $n$.

When $X^TX$ is singular the normal equations still have solutions -- infinitely
many. The fitted values $\hat{\mathbf{y}} = P\mathbf{y}$ are still unique (the
projection is), but the **coefficients** are not: the fit is determined,
$\boldsymbol\beta$ is not. Nearly collinear columns cause the same trouble in a
milder form, which is where
[regularization](11-Regression-II-Logistic-Regularization.qmd) comes in next
lecture.


# The General Linear Model

## The General Linear Model

Another way that the inconsistent (i.e. no exact solution) linear system is often written is to collect
all the residuals into a __residual vector.__ 

Then an exact equation is

$$
\mathbf{y} = X\boldsymbol\beta + \boldsymbol\epsilon,
$$

where $\boldsymbol\epsilon$ is the __residual vector.__

Any equation of this form is referred to as a __linear model.__ 

In this formulation, the goal is to find the $\boldsymbol\beta$ so as to minimize the
__norm__ of $\boldsymbol\epsilon$, i.e., $\Vert\boldsymbol\epsilon\Vert.$

In some cases, one would like to fit data points with something other than a
straight line.  

In cases like this, the matrix equation is still $X\boldsymbol\beta = \mathbf{y}$,
but the specific form of $X$ changes from one problem to the next.


## Least-Squares Fitting of Other Models

In model fitting, the parameters of the model are what is unknown.  

A central question for us is whether the model is _linear_ in its parameters.

For example, is this model linear in its parameters?

$$
y = \beta_0 e^{-\beta_1 x}
$$

It __is not__ linear in its parameters.  


Is this model linear in its parameters?

$$
y = \beta_0 e^{-2 x}
$$


It __is__ linear in its parameters.


---

For a model that is linear in its parameters, an observation is a linear
combination of (arbitrary) known functions.

In other words, a model that is linear in its parameters is

$$
y = \beta_0f_0(x) + \beta_1f_1(x) + \dots + \beta_{p-1}f_{p-1}(x)
$$

where $f_0, \dots, f_{p-1}$ are $p$ known functions (they become the $p$ columns
of $X$) and $\beta_0,\dots,\beta_{p-1}$ are parameters.

## Example

Suppose data points $(x_1, y_1), \dots, (x_n, y_n)$ appear to lie along some
sort of parabola instead of a straight line.  


In [ ]:
ax = plt.figure(figsize = (5, 5)).add_subplot()
centerAxes(ax)
quad = np.array([1, 3, 0.5])
np.random.seed(1)
xquad = -10.0 + 20.0 * np.random.random(100)
yquad = quad[0] + (quad[1] * xquad) + (quad[2] * xquad * xquad) + 2*np.random.randn(100)
ax.plot(xquad, yquad, 'ro', markersize = 4)
plt.show()

As a result, we wish to approximate the data by an equation of the form

$$
y = \beta_0 + \beta_1x + \beta_2x^2.
$$

Let's describe the linear model that produces a "least squares fit" of the data by the equation.


## Solution
  
The ideal relationship is $y = \beta_0 + \beta_1x + \beta_2x^2.$

Suppose the actual values of the parameters are $\beta_0, \beta_1, \beta_2.$  
Then the coordinates of the first data point satisfy the equation

$$
y_1 = \beta_0 + \beta_1x_1 + \beta_2x_1^2 + \epsilon_1,
$$

where $\epsilon_1$ is the residual error between the observed value $y_1$ and
the predicted $y$-value.

Each data point determines a similar equation:

$$
\begin{align*}
y_1 &= \beta_0 + \beta_1x_1 + \beta_2x_1^2 + \epsilon_1, \\
y_2 &= \beta_0 + \beta_1x_2 + \beta_2x_2^2 + \epsilon_2, \\
&\vdots \\
y_n &= \beta_0 + \beta_1x_n + \beta_2x_n^2 + \epsilon_n.
\end{align*}
$$

---


Clearly, this system can be written as $\mathbf{y} = X\boldsymbol\beta + \boldsymbol\epsilon.$

$$
\begin{bmatrix}
y_1\\
y_2\\
\vdots\\
y_n
\end{bmatrix} = 
\begin{bmatrix}1&x_1&x_1^2\\
1&x_2&x_2^2\\
\vdots&\vdots&\vdots\\
1&x_n&x_n^2
\end{bmatrix} 
\begin{bmatrix}
\beta_0\\\
\beta_1\\\
\beta_2
\end{bmatrix}
 + 
 \begin{bmatrix}
 \epsilon_1\\
 \epsilon_2\\
 \vdots\\
 \epsilon_n
 \end{bmatrix}
 $$

Let's solve for $\boldsymbol\beta$, writing the formula out literally:

In [ ]:
#| code-fold: false
n = np.shape(xquad)[0]
X = np.array([np.ones(n), xquad, xquad**2]).T

# Solve for beta
beta = np.linalg.inv(X.T @ X) @ X.T @ yquad

And plot the results.

In [ ]:
ax = ut.plotSetup(-10, 10, -10, 20)
ut.centerAxes(ax)
xplot = np.linspace(-10, 10, 50)
yestplot = beta[0] + beta[1] * xplot + beta[2] * xplot**2
ax.plot(xplot, yestplot, 'b-', lw=2)
ax.plot(xquad, yquad, 'ro', markersize=4)
plt.show()

**Don't actually invert $X^TX$**

`np.linalg.inv(X.T @ X) @ X.T @ y` mirrors the formula and is fine for a first
look, but forming the inverse is numerically unstable when columns of $X$ are
nearly collinear -- small rounding errors in $X^TX$ become large errors in
$\hat{\boldsymbol\beta}$. From here on we use `np.linalg.lstsq(X, y)`, which
solves the same least-squares problem stably (via a QR or SVD factorization);
`np.linalg.solve(X.T @ X, X.T @ y)` is the other common choice.

The refresher's [Least squares beware](A7-Linear-Algebra-Refresher.qmd#least-squares-beware)
page has a three-line example where the inverse gets the answer wrong, and
[Lecture 11](11-Regression-II-Logistic-Regularization.qmd) shows what
near-collinearity does to the coefficients.


---


Now let's try a different model.

$$
\begin{bmatrix}
y_1\\
y_2\\
\vdots\\
y_n
\end{bmatrix} = 
\begin{bmatrix}1&\log(x_1)\\
1&\log(x_2)\\
\vdots&\vdots\\
1&\log(x_n)
\end{bmatrix} 
\begin{bmatrix}
\beta_0\\\
\beta_1
\end{bmatrix}
+ 
\begin{bmatrix}
\epsilon_1\\
\epsilon_2\\
\vdots\\
\epsilon_n
\end{bmatrix}
 $$

In [ ]:
#| code-fold: false
n = np.shape(xlog)[0]
X = np.array([np.ones(n), np.log(xlog)]).T

# Solve for beta, stably this time
beta, *_ = np.linalg.lstsq(X, ylog, rcond=None)

And plot the results.

In [ ]:
# 
# plot the results
#
ax = ut.plotSetup(0, 10, -5, 15)
ut.centerAxes(ax)
xplot = np.linspace(0.05, 10, 200)
yestplot = beta[0] + beta[1] * np.log(xplot)
ax.plot(xplot,yestplot,'b-',lw=2)
ax.plot(xlog,ylog,'ro',markersize=4)
plt.show()

## Multiple Regression

Suppose an experiment involves two independent variables -- say, $u$ and $v$, --
and one dependent variable, $y$.  

A simple equation for predicting $y$ from $u$ and $v$ has the form

$$
y = \beta_0 + \beta_1 u + \beta_2 v.
$$

Since there is more than one independent variable, this is called
__multiple regression.__


A more general prediction equation might have the form

$$
y = \beta_0 + \beta_1 u + \beta_2 v + \beta_3u^2 + \beta_4 uv + \beta_5 v^2.
$$

A least squares fit to equations like this is called a __trend surface.__

In general, a linear model will arise whenever $y$ is to be predicted by an equation of the form

$$
y = \beta_0f_0(u,v) + \beta_1f_1(u,v) + \cdots + \beta_{p-1}f_{p-1}(u,v),
$$

with $f_0,\dots,f_{p-1}$ any sort of known functions and $\beta_0,\dots,\beta_{p-1}$ unknown weights.

## Example

Let's take an example. Here are a set of points in $\mathbb{R}^3$:

In [ ]:
#| fig-align: center
ax = ut.plotSetup3d(-7, 7, -7, 7, -10, 10, figsize = (5, 5))
v = [4.0, 4.0, 2.0]
u = [-4.0, 3.0, 1.0]
npts = 50

# set locations of points that fall within x,y
np.random.seed(1)
xc = -7.0 + 14.0 * np.random.random(npts)
yc = -7.0 + 14.0 * np.random.random(npts)
A = np.array([u,v]).T

# project these points onto the plane
P = A @ np.linalg.inv(A.T @ A) @ A.T
coords = P @ np.array([xc, yc, np.zeros(npts)])
coords[2] += np.random.randn(npts)
ax.plot(coords[0], coords[1], 'ro', zs=coords[2], markersize = 6)
plt.show()

## Example

In geography, local models of terrain are constructed from data

$$
(u_1, v_1, z_1), \dots, (u_n, v_n, z_n),
$$

where $u_j, v_j$, and $z_j$ are latitude, longitude, and altitude, respectively.

Let's describe the linear models that gives a least-squares fit to such data.  
The solution is called the least-squares _plane._


## Solution

The model is $z = \beta_0 + \beta_1 u + \beta_2 v + \epsilon$: altitude as a
linear function of latitude and longitude. We expect the data to satisfy these
equations:

$$
\begin{align*}
z_1 &= \beta_0 + \beta_1 u_1 + \beta_2 v_1 + \epsilon_1, \\
z_2 &= \beta_0 + \beta_1 u_2 + \beta_2 v_2 + \epsilon_2, \\
\vdots \\
z_n &= \beta_0 + \beta_1 u_n + \beta_2 v_n + \epsilon_n.
\end{align*}
$$

This system has the matrix form $\mathbf{z} = X\boldsymbol\beta + \boldsymbol\epsilon,$
with an $n \times 3$ design matrix $X$:

$$
\mathbf{z} =
\begin{bmatrix}z_1\\
z_2\\
\vdots\\
z_n
\end{bmatrix},\;\;
X =
\begin{bmatrix}1&u_1&v_1\\
1&u_2&v_2\\
\vdots&\vdots&\vdots\\
1&u_n&v_n
\end{bmatrix},\;\;
\boldsymbol\beta=
\begin{bmatrix}
\beta_0\\
\beta_1\\
\beta_2
\end{bmatrix},\;\;
\boldsymbol\epsilon =
\begin{bmatrix}
\epsilon_1\\
\epsilon_2\\
\vdots\\
\epsilon_n
\end{bmatrix}.
$$

---


In [ ]:
#| fig-align: center
u_obs, v_obs, z_obs = coords

# design matrix [1, u, v] and the least-squares plane
X_plane = np.column_stack([np.ones(npts), u_obs, v_obs])
beta_plane, *_ = np.linalg.lstsq(X_plane, z_obs, rcond=None)
z_hat = X_plane @ beta_plane

ax = ut.plotSetup3d(-7, 7, -7, 7, -10, 10, figsize = (7, 7))
ax.set_xlabel('$u$', size=15)
ax.set_ylabel('$v$', size=15)
ax.set_zlabel('$z$', size=15)

# the fitted plane ...
uu, vv = np.meshgrid(np.linspace(-7, 7, 10), np.linspace(-7, 7, 10))
ax.plot_surface(uu, vv, beta_plane[0] + beta_plane[1] * uu + beta_plane[2] * vv,
                color='green', alpha=0.3)
# ... and the residual from each point to it
for ui, vi, zi, zh in zip(u_obs, v_obs, z_obs, z_hat):
    ax.plot([ui, ui], [vi, vi], [zi, zh], 'b-', lw=1)
ax.plot(u_obs, v_obs, 'ro', zs=z_obs, markersize=6)
plt.show()

The green surface is the fitted plane $\hat z = \hat\beta_0 + \hat\beta_1 u + \hat\beta_2 v$;
the blue segments are the residuals it minimizes (in the sum of squares).

This example shows that the linear model for multiple regression has the same
form as the model for the simple regression in the earlier examples.

We can see that the general principle is the same across all the different
kinds of linear models.

Once $X$ is defined properly, the normal equations for $\boldsymbol\beta$ have the
same matrix form, no matter how many variables are involved.

Thus, for any linear model where $X^TX$ is invertible, the least squares estimate is

$$
\hat{\boldsymbol\beta} = (X^TX)^{-1}X^T\mathbf{y}.
$$



## Measuring the fit of a regression model: $R^2$

Given any $X$ and $\mathbf{y}$, the above algorithm will produce an output
$\hat{\boldsymbol\beta}$.

But how do we know whether the data is in fact well described by the model?

The most common measure of fit is $R^2$.

$R^2$ measures the __fraction of the variance__ of $\mathbf{y}$ that can be
explained by the model $X\hat{\boldsymbol\beta}$.

The variance of $\mathbf{y}$ is 

$$
\text{Var}(\mathbf{y}) =\frac{1}{n} \sum_{i=1}^n \left(y_i-\overline{y}\right)^2,
$$

where $\overline{y}=\frac{1}{n}\sum_{i=1}^ny_i$.



For any given $n$, we can equally work with just 
$$
\sum_{i=1}^n \left(y_i-\overline{y}\right)^2,
$$

which is called the __Total Sum of Squares__ (TSS).

## Total Sum of Squares

Now to measure the quality of fit of a model, we break TSS down into two components. 

For any given $\mathbf{x}_i$, the prediction made by the model is 
$\hat{y_i} = \mathbf{x}_i^T\boldsymbol\beta$.

We can break the total sum of squares into two parts:

* the residual $\epsilon_i$ is $y_i - \hat{y_i}$, and 
* the part that the model "explains" is $\hat{y_i} - \overline{y}.$

## RSS and ESS

So we define the Residual Sum of Squares (RSS) as:

$$
\text{RSS} = \sum_{i=1}^n \left(y_i-\hat{y_i}\right)^2,
$$

and the Explained Sum of Squares (ESS) as:

$$
\text{ESS} = \sum_{i=1}^n \left(\hat{y_i}-\overline{y}\right)^2,
$$

One can show that the total sum of squares is exactly equal to the sum of
squares of the residuals plus the sum of squares of the explained part.

In other words:

$$
\text{TSS} = \text{RSS} + \text{ESS}.
$$



## Proof that TSS = RSS + ESS

**Given:**

- TSS = $\sum_{i=1}^n (y_i - \bar{y})^2$ (Total Sum of Squares)
- RSS = $\sum_{i=1}^n (y_i - \hat{y}_i)^2$ (Residual Sum of Squares)
- ESS = $\sum_{i=1}^n (\hat{y}_i - \bar{y})^2$ (Explained Sum of Squares)

**Proof:**

Start by decomposing the deviation from the mean:
$$y_i - \bar{y} = (y_i - \hat{y}_i) + (\hat{y}_i - \bar{y})$$

Square both sides:
$$
\begin{align}
(y_i - \bar{y})^2 &= [(y_i - \hat{y}_i) + (\hat{y}_i - \bar{y})]^2\\
&= (y_i - \hat{y}_i)^2 + 2(y_i - \hat{y}_i)(\hat{y}_i - \bar{y}) + (\hat{y}_i - \bar{y})^2
\end{align}
$$

Sum over all $i$:
$$
\sum_{i=1}^n (y_i - \bar{y})^2 = \sum_{i=1}^n (y_i - \hat{y}_i)^2 + 2\sum_{i=1}^n (y_i - \hat{y}_i)(\hat{y}_i - \bar{y}) + \sum_{i=1}^n (\hat{y}_i - \bar{y})^2
$$

Which gives us:
$$
\text{TSS} = \text{RSS} + 2\sum_{i=1}^n (y_i - \hat{y}_i)(\hat{y}_i - \bar{y}) + \text{ESS}
$$

**The key step:** We need to show that the cross-term equals zero:
$$
\sum_{i=1}^n (y_i - \hat{y}_i)(\hat{y}_i - \bar{y}) = 0
$$

**Why is this zero?**

For OLS regression **with an intercept**, the residuals $(y_i - \hat{y}_i)$ satisfy two orthogonality conditions:

1. $\sum_{i=1}^n (y_i - \hat{y}_i) = 0$ (residuals sum to zero)
2. $\sum_{i=1}^n (y_i - \hat{y}_i)\hat{y}_i = 0$ (residuals are orthogonal to predictions)

Both are the normal equations $X^T\mathbf{e} = \mathbf{0}$ read row by row: the row
for the column of 1s says the residuals sum to zero (condition 1), and because
$\hat{\mathbf{y}} = X\hat{\boldsymbol\beta}$ is a combination of the columns of $X$,
$\mathbf{e}$ is orthogonal to it too (condition 2). The proofs below spell this out.

Using condition (2):
$$
\sum_{i=1}^n (y_i - \hat{y}_i)(\hat{y}_i - \bar{y}) = \sum_{i=1}^n (y_i - \hat{y}_i)\hat{y}_i - \bar{y}\sum_{i=1}^n (y_i - \hat{y}_i) = 0 - \bar{y} \cdot 0 = 0
$$

Therefore: **TSS = RSS + ESS** ✓

**Geometric Intuition:**

This decomposition is a Pythagorean theorem in high-dimensional space! The vectors $(y_i - \bar{y})$, $(y_i - \hat{y}_i)$, and $(\hat{y}_i - \bar{y})$ form a right triangle because the residuals are orthogonal to the fitted values.

#### Why residuals sum to exactly zero (with intercept)

Condition 1 is satisfied **exactly** (not just approximately), but this is **only true when the model includes an intercept term**.

When you include an intercept $\beta_0$, the normal equations $X^TX\boldsymbol\beta = X^T\mathbf{y}$ include a row corresponding to the column of 1s in the design matrix.

#### Proof

Consider a linear model with an intercept. The design matrix has a column of 1s:

$$
X = \begin{bmatrix}
1 & x_{11} & x_{12} & \cdots & x_{1p} \\
1 & x_{21} & x_{22} & \cdots & x_{2p} \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & x_{n1} & x_{n2} & \cdots & x_{np}
\end{bmatrix}, \quad
\boldsymbol\beta = \begin{bmatrix}
\beta_0 \\
\beta_1 \\
\vdots \\
\beta_p
\end{bmatrix}, \quad
\mathbf{y} = \begin{bmatrix}
y_1 \\
y_2 \\
\vdots \\
y_n
\end{bmatrix}
$$

#### The Normal Equations

The OLS solution satisfies the normal equations:
$$
X^T X \boldsymbol\beta = X^T \mathbf{y}
$$

#### Looking at $X^T$

The transpose of $X$ is:
$$
X^T = \begin{bmatrix}
1 & 1 & 1 & \cdots & 1 \\
x_{11} & x_{21} & x_{31} & \cdots & x_{n1} \\
x_{12} & x_{22} & x_{32} & \cdots & x_{n2} \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
x_{1p} & x_{2p} & x_{3p} & \cdots & x_{np}
\end{bmatrix}
$$

**The first row is all 1s!**

#### The First Normal Equation

The normal equations give us a system of $(p+1)$ equations. The **first equation** comes from multiplying the first row of $X^T$ with both sides:

Left side: First row of $X^T$ times $X\boldsymbol\beta$
$$
[1, 1, 1, \ldots, 1] \cdot X\boldsymbol\beta = \sum_{i=1}^n (X\boldsymbol\beta)_i = \sum_{i=1}^n \hat{y}_i
$$

Right side: First row of $X^T$ times $\mathbf{y}$
$$
[1, 1, 1, \ldots, 1] \cdot \mathbf{y} = \sum_{i=1}^n y_i
$$

#### The Result

Therefore, the first normal equation is:
$$
\boxed{\sum_{i=1}^n \hat{y}_i = \sum_{i=1}^n y_i}
$$

This is **exact**, not approximate. It's a direct consequence of having the column of 1s in $X$.

#### Interpretation

Dividing by $n$:
$$
\bar{\hat{y}} = \bar{y}
$$

The mean of the fitted values equals the mean of the observed values. This is why, with an intercept, the regression line always passes through the point $(\bar{x}, \bar{y})$.

**End of proof**


#### Without an intercept

If you force the regression through the origin (no $\beta_0$ term), then:

- Condition 1 is **NOT** satisfied - residuals don't sum to zero
- The TSS = RSS + ESS decomposition **doesn't hold**
- $R^2$ can even be negative!


#### Key takeaway

For standard OLS with intercept:
- $\sum (y_i - \hat{y}_i) = 0$ ✓ (exactly)
- $\sum (y_i - \hat{y}_i)\hat{y}_i = 0$ ✓ (exactly, from normal equations)

Both are mathematical properties, not approximations!


## $R^2$, cont.

Now, a good fit is one in which the model explains a large part of the variance
of $\mathbf{y}$.  

So the measure of fit $R^2$ is defined as:

$$
R^2 = \frac{\text{ESS}}{\text{TSS}} 
= 1-\frac{\text{RSS}}{\text{TSS}}
= 1 - \frac{\sum_{i=1}^n \left(y_i-\hat{y_i}\right)^2}{\sum_{i=1}^n \left(y_i-\overline{y}\right)^2}.
$$

As a result, $0\leq R^2\leq 1$ -- __in-sample, for a model with an intercept__.
That is what makes $\text{TSS} = \text{RSS} + \text{ESS}$ hold. Computed on
held-out data (as `r2_score` does for Ames below) RSS can exceed TSS, and $R^2$
can be negative: the model predicts worse than the constant $\overline{y}$.

This is more specifically called the __centered__ $R^2$; it assumes an
intercept. We will meet the __uncentered__ variant in a moment, when a
software package quietly switches to it.



The closer the value of $R^2$ is to $1$ the better the fit of the regression:

* small values of RSS imply that the residuals are small and therefore we have a better fit.

$R^2$ is called the __coefficient of determination.__  

It tells us "how well does the model predict the data?"

## $R^2$ and Pearson's $r$

$R^2$ is related to, but not the same thing as, Pearson's __correlation
coefficient__ $r$.

* For __simple__ regression (one feature plus an intercept),
  $R^2 = r^2$ exactly, where $r = \text{corr}(x, y)$. That is where the
  notation comes from, and why people sometimes write $r^2$.
* In general (any number of features, with an intercept, in-sample),
  $$R^2 = \text{corr}(\mathbf{y}, \hat{\mathbf{y}})^2,$$
  the squared correlation between the observations and the model's
  __predictions__.
* So $R^2$ is a correlation squared, but between $\mathbf{y}$ and
  $\hat{\mathbf{y}}$, not between two raw variables. Two features can each be
  correlated with $y$ while the model built from them predicts it poorly; to
  judge predictions against ground truth, use $R^2$ (and use it on held-out
  data).


# OLS in Practice

## OLS in Practice

First, we'll look at a test case on synthetic data. We'll use Scikit-Learn's
`make_regression` function.

In [ ]:
#| code-fold: false
from sklearn import datasets
X, y = datasets.make_regression(n_samples=100, n_features=20, n_informative=5, bias=0.1, noise=30, random_state=1)

And then use the `statsmodels` package to fit the model.

In [ ]:
#| code-fold: false
import statsmodels.api as sm

# statsmodels does not add an intercept for you: prepend a column of ones
X_with_intercept = sm.add_constant(X)

model = sm.OLS(y, X_with_intercept)
results = model.fit()

---

And print the summary of the results.

In [ ]:
print(results.summary())

The $R^2$ value is very good.   We can see that the linear model does a very good
job of predicting the observations $y_i$.

**Uncentered $R^2$**

`sm.OLS` does __not__ add an intercept for you -- hence `sm.add_constant`. Fit
`sm.OLS(y, X)` on the raw `X` instead and the summary switches to the
__uncentered__ $R^2$,

$$
R^2 (\text{uncentered}) = 1 - \frac{\sum_{i=1}^n \left(y_i-\hat{y_i}\right)^2}{\sum_{i=1}^n y_i^2},
$$

and says so in a footnote. Without a constant column the residuals need not sum
to zero, $\text{TSS} = \text{RSS} + \text{ESS}$ fails and the centered $R^2$
can be negative, so statsmodels divides by $\sum y_i^2$ instead of
$\sum (y_i - \overline{y})^2$. The uncentered number is almost always larger
and is not comparable to the centered one. Try it.




## Aside: Summary Statistics

Here is what all the statistics in the summary table mean. Throughout, $n$ is
the number of observations and $p$ the number of columns of $X$ __including the
constant__ (so $p = 21$ above). When the model has no constant, statsmodels
reports the uncentered versions of the first three statistics (see the callout
above), with $p$ in place of $p - 1$.

1. **R-squared:**

* Definition: the fraction of the variance of the dependent variable
  explained by the model -- the centered $R^2$ defined above.
* Formula:
  $R^2 = 1 - \frac{\text{RSS}}{\text{TSS}} = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \overline{y})^2}$

where $y_i$ is the actual value, and $\hat{y}_i$ is the predicted value from the regression.

* Interpretation: Higher values (closer to 1) indicate a better in-sample fit.
  It never decreases when a column is added, so it cannot by itself choose
  between models.

2. **Adjusted R-squared:**

* Definition: Adjusts $R^2$ for the number of columns in the model, so that
  adding a useless column is penalized rather than rewarded.
* Formula:
    $R^2_{\text{adj}} = 1 - \left( \frac{(1 - R^2)(n - 1)}{n - p} \right)$

* Interpretation: It is a more conservative measure of fit than the regular $R^2$,
  particularly useful when comparing models with different numbers of predictors.
  Unlike $R^2$, it can fall when a column is added.

3. **F-statistic:**

* Definition: The F-statistic tests whether the overall regression model is
  significant, i.e., whether at least one of the (non-intercept) predictors explains a significant
  amount of the variance in the dependent variable.
* Formula:
    $F = \frac{R^2 / (p - 1)}{(1 - R^2) / (n - p)}$

* Interpretation: A high F-statistic suggests that the model is statistically significant.

4. **Prob (F-statistic):**

* Definition: This is the p-value associated with the F-statistic. It indicates the
  probability that the observed F-statistic would occur if the null hypothesis
  (that none of the predictors are significant) were true.
* Interpretation: A small p-value (typically less than 0.05) suggests that the
  model is statistically significant, meaning that at least one predictor is important.

5. **Log-Likelihood:**

* Definition: The log-likelihood measures the likelihood that the model could
  have produced the observed data. It is a measure of the fit of the model to the data.
* Formula:
    $\mathcal{L} = -\frac{n}{2} \log(2\pi \sigma^2) - \frac{1}{2\sigma^2} \sum (y_i - \hat{y}_i)^2$

where $\sigma^2$ is the variance of the residuals.

* Interpretation: A higher log-likelihood indicates a better fit of the model.

6. **Akaike Information Criterion (AIC):**

* Definition: AIC is a measure of the relative quality of a statistical model for a
  given set of data. It penalizes the likelihood function for adding too many
  parameters.
* Formula:
    $\text{AIC} = 2p - 2 \log(\mathcal{L})$

where $p$ is the number of parameters, and $\mathcal{L}$ is the log-likelihood.

* Interpretation: Lower AIC values suggest a better model fit, but it penalizes for overfitting.

7. **Bayesian Information Criterion (BIC):**

* Definition: Similar to AIC, BIC penalizes models with more parameters, but more
  strongly than AIC. It is used to select among models.

* Formula:
    $\text{BIC} = p \log(n) - 2 \log(\mathcal{L})$

where $p$ is the number of parameters, $n$ is the number of observations, and
$\mathcal{L}$ is the log-likelihood.

* Interpretation: A lower BIC indicates a better model, with stronger penalties for
  models with more parameters.

8. **Coefficient Table:**

* The first two columns are the independent variables and their coefficients. It
  is the $m$ in $y = mx + b$. One unit of change in the variable will affect the
  variable's coefficient's worth of change in the dependent variable, with the
  other variables held fixed. If the
  coefficient is negative, they have an inverse relationship. As one rises, the
  other falls.

* The `std error` is an estimate of the standard deviation of the coefficient, a
  measurement of the amount of variation in the coefficient throughout its data
  points.

* The `t` is related and is a measurement of the precision with which the coefficient
  was measured. A low std error compared to a high coefficient produces a
  high t statistic, which signifies a high significance for your coefficient.

* `P>|t|` is one of the most important statistics in the summary. It uses the
  `t` statistic to produce the `P` value: the probability of seeing a `t`
   statistic at least this extreme if the true coefficient were zero. For example,
   a `P` value of 0.378 says that, if the variable had no effect on the dependent
   variable, a coefficient this far from zero would still appear 37.8% of the time
   by chance. It is not the probability that the variable has no effect. Proper model analysis
   will compare the `P` value to a previously established alpha value, or a
   threshold with which we can apply significance to our coefficient. A common
   alpha is 0.05, which few of our variables pass in this instance.

* `[0.025 and 0.975]` are the lower and upper ends of the 95% confidence interval
  for the coefficient, roughly the estimate plus or minus two standard errors. If
  the interval contains zero, the data cannot distinguish the coefficient from zero.

The coefficient description came from this [blog post](https://medium.com/swlh/interpreting-linear-regression-through-statsmodels-summary-4796d359035a)
which also describes the other statistics in the summary table.


---

Back to the example.

Some of the independent variables may not contribute to the accuracy of
the prediction.

Each coefficient comes with a 95% confidence interval -- the `[0.025  0.975]`
columns of the table. Plotting them side by side makes the table readable at a
glance:

In [ ]:
#| fig-align: center
coef = results.params
CIs = results.conf_int()
names = ['const'] + [f'$x_{{{j}}}$' for j in range(1, X.shape[1] + 1)]  # match statsmodels' x1..x20

fig, ax = plt.subplots(figsize=(10, 4))
ax.errorbar(np.arange(len(coef)), coef,
            yerr=[coef - CIs[:, 0], CIs[:, 1] - coef],
            fmt='o', capsize=3, label='estimate with 95% CI')
ax.axhline(0, color='r', lw=1)
ax.set_xticks(np.arange(len(coef)))
ax.set_xticklabels(names)
ax.set_ylabel('coefficient')
ax.legend()
plt.show()

If a coefficient's interval contains zero, the data cannot distinguish it from
zero, and we cannot assume that there is any relationship between that
independent variable and the observations. Six intervals sit clearly away from
zero -- but `make_regression` was told `n_informative=5`. One of the six is a
false positive: at the 95% level, roughly one useless column in twenty will look
significant by chance, and we have fifteen useless columns.





## Confidence Interval Formula

For coefficient $\beta_j$, the 95% confidence interval is:

$$
\hat{\beta}_j \pm t_{\alpha/2, n-p} \cdot \text{SE}(\hat{\beta}_j)
$$

where:
- $\hat{\beta}_j$ is the estimated coefficient
- $t_{\alpha/2, n-p}$ is the critical value from the t-distribution with $n-p$ degrees of freedom (for 95% CI, $\alpha = 0.05$)
- $\text{SE}(\hat{\beta}_j)$ is the standard error of $\hat{\beta}_j$

#### Calculating the Standard Error

The standard errors come from the **variance-covariance matrix** of the coefficient estimates:

#### Step 1: Estimate residual variance
$$
\hat{\sigma}^2 = \frac{\text{RSS}}{n-p} = \frac{\sum_{i=1}^n (y_i - \hat{y}_i)^2}{n-p}
$$

where $n$ = number of observations, $p$ = number of parameters (including intercept)

#### Step 2: Compute variance-covariance matrix
$$
\text{Var}(\hat{\beta}) = \hat{\sigma}^2 (X^T X)^{-1}
$$

This is a $p \times p$ matrix where:
- Diagonal elements are variances of each coefficient
- Off-diagonal elements are covariances between coefficients

##### Step 3: Extract standard errors
$$
\text{SE}(\hat{\beta}_j) = \sqrt{[(X^T X)^{-1}]_{jj} \cdot \hat{\sigma}^2}
$$

The standard error is the square root of the $j$-th diagonal element of the variance-covariance matrix.

#### Interpretation

In the statsmodels summary, the `[0.025, 0.975]` columns are exactly these bounds:
- Lower bound: $\hat{\beta}_j - t_{0.025, n-p} \cdot \text{SE}(\hat{\beta}_j)$
- Upper bound: $\hat{\beta}_j + t_{0.025, n-p} \cdot \text{SE}(\hat{\beta}_j)$

If this interval contains zero, the coefficient is not statistically significant - we can't distinguish it from no effect.

#### Why $(X^T X)^{-1}$?

This matrix captures the geometry of your design:
- Larger values → more uncertainty (wider CIs)
- Happens when: features are correlated (multicollinearity) or you have fewer data points


---

In other words, if the confidence interval for the parameter includes zero, the
associated independent variable may not have any predictive value.


In [ ]:
print('Confidence Intervals: {}'.format(results.conf_int()))
print('Parameters: {}'.format(results.params))

Find the independent variables that are not significant.

In [ ]:
CIs = results.conf_int()
notSignificant = (CIs[:,0] < 0) & (CIs[:,1] > 0)
notSignificant[0] = False   # always keep the intercept column
notSignificant

Let's look at the shape of significant variables.

In [ ]:
Xsignif = X_with_intercept[:,~notSignificant]
Xsignif.shape

---

A tempting next step is to drop the columns whose interval covers zero and
refit:

In [ ]:
model = sm.OLS(y, Xsignif)
results = model.fit()
print(results.summary())

Here that is nearly harmless -- `make_regression` built $y$ from five
informative columns, and we kept those five plus one impostor. In general,
though, be careful: choosing the features __and__ judging the fit on the
__same__ data is the [Lecture 7](07-Generalization.qmd) anti-pattern. The
refit looks better than it is, because the selection step already used the
noise in this sample (that is exactly how the impostor got in).

* The honest version checks the smaller model on data it has not seen -- which
  is exactly what we do for the Ames houses next.
* Dropping a column is also an all-or-nothing choice. Next lecture's
  __regularization__ replaces manual pruning with a penalty that shrinks
  coefficients toward zero smoothly, and lets held-out data pick how hard to
  shrink.


## Real Data: House Prices in Ames, Iowa

Let's see how powerful multiple regression can be on a real-world example.

A typical application of linear models is predicting house prices.

Linear models have been used for this problem for decades, and when a municipality
does a value assessment on your house, they typically use a linear model.

We can consider various measurable attributes of a house (its "features") as the
independent variables, and the most recent sale price of the house as the
dependent variable.

---

For our case study, we will use the features:

* Lot Area (sq ft), 
* Gross Living Area (sq ft), 
* Number of Fireplaces, 
* Number of Full Baths, 
* Number of Half Baths, 
* Garage Area (sq ft), 
* Basement Area (sq ft)

So our design matrix will have 8 columns (including the constant for the intercept):

$$
X\beta = \mathbf{y},
$$

and it will have one row for each house in the data set, with $y$ the sale price
of the house.


We will use data from housing sales in Ames, Iowa from 2006 to 2009:

![Ames Iowa](figs/ames-iowa-downtown.jpeg)

[Tim Kiser (w:User:Malepheasant)](https://commons.wikimedia.org/wiki/File:Ames_Iowa_Main_Street.jpg)
[CC BY-SA 2.5](https://creativecommons.org/licenses/by-sa/2.5)
via Wikimedia Commons

---

In [ ]:
df = pd.read_csv('data/ames-housing-data/train.csv')

In [ ]:
df[['LotArea', 'GrLivArea', 'Fireplaces', 'FullBath', 'HalfBath', 'GarageArea', 'TotalBsmtSF', 'SalePrice']].head()

---

Some things to note here:

* House prices are in dollars
* Areas are in square feet
* Rooms are in counts

Do we have scaling concerns here?  

No, because each feature will get its own $\beta$, which will correct for the
scaling differences between different units of measure.

In [ ]:
X_no_intercept = df[['LotArea', 'GrLivArea', 'Fireplaces', 'FullBath', 'HalfBath', 'GarageArea', 'TotalBsmtSF']]
X_intercept = sm.add_constant(X_no_intercept)
y = df['SalePrice'].values

**Note**

Note that removing the intercept will cause the $R^2$ to go up, which is 
counter-intuitive.  The reason is explained 
[here](https://stats.stackexchange.com/questions/26176/removal-of-statistically-significant-intercept-term-increases-r2-in-linear-mo/26205#26205)
-- but amounts to the fact that
the formula for $R^2$ with/without an intercept is different.



Let's split the data into training and test sets.

In [ ]:
from sklearn import utils, model_selection
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X_intercept, y, test_size = 0.5, random_state = 0)

Fit the model to the training data.

In [ ]:
model = sm.OLS(y_train, X_train)
results = model.fit()
print(results.summary())

---

We see that we have:

* $\beta_0$: Intercept of -\$42,850
* $\beta_1$: Marginal value of one square foot of Lot Area: \$0.23 
    * but __NOTICE__ - this coefficient is not statistically different from zero!
* $\beta_2$: Marginal value of one square foot of Gross Living Area: \$48 
* $\beta_3$: Marginal value of one additional fireplace: \$10,890
* $\beta_4$: Marginal value of one additional full bath: \$14,900
* $\beta_5$: Marginal value of one additional half bath: \$15,600
* $\beta_6$: Marginal value of one square foot of Garage Area: \$99
* $\beta_7$: Marginal value of one square foot of Basement Area: \$62


Why is a *half* bath (\$15,600) worth more than a *full* bath (\$14,900)?

* Each $\beta_j$ holds the __other six features fixed__: $\beta_5$ is the value
  of a half bath *given* the living area, basement, garage and so on.
* Half baths co-occur with bigger, newer houses, and the coefficient absorbs
  some of what those features do not capture. Add or remove a correlated
  feature and the other coefficients __move__.
* A coefficient is a __conditional slope__, not a price tag -- and not a causal
  claim. Adding a half bath to your house will not raise its price by \$15,600.


---

Is our model doing a good job?

The summary table offers many statistics, but two pictures tell us more: the
predictions against the ground truth, and then the residuals against the
fitted values.

For each house we compute its predicted sale value according to our model:

$$
\hat{\mathbf{y}} = X\hat{\boldsymbol\beta}
$$

In [ ]:
from sklearn.metrics import r2_score

fig, (ax1, ax2) = plt.subplots(1,2,sharey = 'row', figsize=(12, 5))
y_oos_predict = results.predict(X_test)
r2_test = r2_score(y_test, y_oos_predict)
ax1.scatter(y_test, y_oos_predict, s = 8)
ax1.set_xlabel('True Price')
ax1.set_ylabel('Predicted Price')
ax1.plot([0,500000], [0,500000], 'r-')
ax1.axis('equal')
ax1.set_ylim([0, 500000])
ax1.set_xlim([0, 500000])
ax1.set_title(f'Out of Sample Prediction, $R^2$ is {r2_test:0.3f}')
#
y_is_predict = results.predict(X_train)
ax2.scatter(y_train, y_is_predict, s = 8)
r2_train = r2_score(y_train, y_is_predict)
ax2.set_xlabel('True Price')
ax2.plot([0,500000],[0,500000],'r-')
ax2.axis('equal')
ax2.set_ylim([0,500000])
ax2.set_xlim([0,500000])
ax2.set_title(f'In Sample Prediction, $R^2$ is {r2_train:0.3f}')
plt.show()

We see that the model does a reasonable job in the middle of the price range,
where most of the houses are.

At both ends the predictions are pulled toward the mean: cheap houses are
__over__-predicted and expensive ones __under__-predicted. That is not a bug in
this particular fit. $\hat{\mathbf{y}}$ is a projection of $\mathbf{y}$, so it
is the shorter vector and $\text{Var}(\hat{\mathbf{y}}) \le \text{Var}(\mathbf{y})$:
fitted values are always less spread out than the truth. This is Galton's
regression to the mean, now falling out of the algebra.

Note that the $R^2$ on the (held out) test data is 0.610, against 0.759 in
sample. We are not doing as well on test data as on training data (somewhat to
be expected).


$R^2$ compresses the fit into one number; the residuals
$\mathbf{e} = \mathbf{y} - \hat{\mathbf{y}}$ show *where* the model is wrong.
Plot them against the fitted values, again for the held-out houses:

In [ ]:
#| fig-align: center
resid_test = y_test - y_oos_predict

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.scatter(y_oos_predict, resid_test, s=8)
ax.axhline(0, color='r')
ax.set_xlabel('Fitted (predicted) price')
ax.set_ylabel('Residual: true minus predicted')
ax.set_title('Residuals vs fitted, held-out houses')
plt.show()

A healthy fit shows a flat, even band around zero. This one __fans out__: the
spread of the residuals grows with the predicted price, so the model is far less
reliable for expensive houses than for cheap ones (__heteroscedasticity__).
Between roughly \$300,000 and \$500,000 the residuals are mostly positive --
the under-prediction of expensive houses we saw above.

Then there are the two points on the far right: houses the model prices at
about \$550,000 and \$850,000 that sold for hundreds of thousands less. These
are the famous Ames outliers -- two enormous houses (over 4,000 sq ft of living
area) sold under unusual *partial* sale conditions. Points far out in feature
space with huge residuals: a __high-leverage__ problem, and the dataset's own
documentation recommends removing them. The scatter above hid them (they are
off its axes); the residual plot does not.


The standard fix for the fan is to model $\log(\text{SalePrice})$ rather than
the price itself: errors become *proportional* rather than additive, which is
more plausible for prices and flattens the fan. The cost is that coefficients
then read as percentage changes, not dollars.


Make this plot every time. Three patterns to look for: a __bow__ (a missing
nonlinear term), a __fan__ (spread that changes with the fit), and a lone point
far out in $x$ (__high leverage__, a large $P_{ii}$) that can tilt the whole
fit. You will practise all three in the activity.


For a better model, we'd also want to consider more features of each house, and
perhaps some additional functions such as polynomials as components of our model.

## Recap

* Linear models are a powerful tool for prediction and inference
* Least squares is a projection: the normal equations say the residual is
  orthogonal to every column of $X$
* The $R^2$ statistic provides a simple measure of fit; judge it on held-out data
* Coefficients are conditional slopes, each read with the other features held fixed
* Plot the residuals against the fitted values: a bow, a fan or a lone
  high-leverage point says the model is wrong in a way $R^2$ cannot show
* Careful use of the model is required to avoid overfitting
